Согласие LLM-судей с экспертами: каппа Коэна.....


In [1]:
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'evaluations/human_and_model_validation'
os.chdir(_NOTEBOOK_DIR)


In [2]:
import json
import re
from pathlib import Path
import numpy as np
import openpyxl
import pandas as pd
from sklearn.metrics import cohen_kappa_score
HERE = Path.cwd()
BLIND_CSV = HERE / 'blind_protocols_60.csv'
LLM_JUDGES = {'gpt4.1': HERE / 'gpt41_judging' / 'judge_gpt41' / 'blind_protocols_60_gpt41_judge.csv', 'gpt4.1-mini': HERE / 'gpt41_mini_judging' / 'blind_protocols_60_gpt41_mini_judge.csv'}
EXPERT_FILES = {path.name.removesuffix('_blind_protocols_60_ru_final.xlsx'): path for path in sorted((HERE / 'expert_ratings').glob('*_blind_protocols_60_ru_final.xlsx'))}
KEY = ['case_id', 'protocol', 'criterion']
N_BOOT = 5000
SEED = 20260919
ROUND = {'κ': 3, 'согласие, %': 1}
print('LLM-судьи:', list(LLM_JUDGES))
print('Эксперты: ', list(EXPERT_FILES))


LLM-судьи: ['gpt4.1', 'gpt4.1-mini']
Эксперты:  ['expert_01', 'expert_02', 'expert_03', 'expert_04']


In [3]:
def read_llm(path: Path) -> pd.DataFrame:
    judged = pd.read_csv(path, sep=';', encoding='utf-8-sig')
    rows = [(row.case_id, int(row.blind_slot), int(item['id']), int(item['points']), int(bool(item['met']))) for row in judged.itertuples() for item in json.loads(row.judge_details)]
    return pd.DataFrame(rows, columns=KEY + ['points', 'met'])
llm = {name: read_llm(path) for name, path in LLM_JUDGES.items()}
for name, df in llm.items():
    print(f"{name:<12} {len(df)} оценок, {df['case_id'].nunique()} кейсов")


gpt4.1       3408 оценок, 60 кейсов
gpt4.1-mini  3408 оценок, 60 кейсов


In [4]:
case_by_number = dict(enumerate(pd.read_csv(BLIND_CSV, sep=';', encoding='utf-8-sig')['case_id'], start=1))
number_by_case = {case_id: number for number, case_id in case_by_number.items()}
CASE_HEADER = re.compile('CASE\\s+(\\d+)\\s+·')
PROTOCOL_HEADER = re.compile('КРИТЕРИИ · ПРОТОКОЛ\\s+(\\d+)')
VERDICTS = {'да': 1, 'нет': 0}

def read_expert(path: Path) -> pd.DataFrame:
    sheet = openpyxl.load_workbook(path, read_only=True)['Review']
    rows, case, protocol = ([], None, None)
    for row in sheet.iter_rows(max_col=4, values_only=True):
        first, points, _, verdict = (tuple(row) + (None,) * 4)[:4]
        label = '' if first is None else str(first)
        if (match := CASE_HEADER.match(label)):
            case, protocol = (int(match[1]), None)
            continue
        if (match := PROTOCOL_HEADER.match(label)):
            protocol = int(match[1])
            continue
        if label.startswith('СВОДНАЯ'):
            protocol = None
            continue
        if protocol is None:
            continue
        try:
            criterion = int(float(first))
        except (TypeError, ValueError):
            continue
        text = verdict.strip().lower() if isinstance(verdict, str) else verdict
        if text in (None, ''):
            met = np.nan
        elif text in VERDICTS:
            met = VERDICTS[text]
        else:
            raise ValueError(f'{path.name}: кейс {case}, протокол {protocol}, критерий {criterion}: {verdict!r}')
        rows.append((case_by_number[case], protocol, criterion, int(float(points)), met))
    return pd.DataFrame(rows, columns=KEY + ['points', 'met'])
experts = {name: read_expert(path) for name, path in EXPERT_FILES.items()}


In [5]:
grid = llm['gpt4.1'][KEY + ['points']].sort_values(KEY).reset_index(drop=True)
for name, df in {**llm, **experts}.items():
    same = df[KEY + ['points']].sort_values(KEY).reset_index(drop=True).equals(grid)
    assert same, f'{name}: сетка оценок или баллы не совпадают'
ratings = grid.set_index(KEY)
for name, df in {**llm, **experts}.items():
    ratings[name] = df.set_index(KEY)['met']
ratings = ratings.reset_index()
LLMS, EXPERTS = (list(llm), list(experts))
print(f'Сетка: {len(grid)} оценок, одинаковая у всех {len(LLMS) + len(EXPERTS)} судей')

def as_ranges(numbers) -> str:
    numbers = pd.Series(sorted(numbers))
    runs = numbers.groupby(numbers.diff().ne(1).cumsum())
    return ', '.join((f'{r.iloc[0]}–{r.iloc[-1]}' if len(r) > 1 else str(r.iloc[0]) for _, r in runs))
coverage = {}
for name in EXPERTS:
    started = ratings.groupby('case_id')[name].transform(lambda s: s.notna().any())
    coverage[name] = {'оценок': int(ratings[name].notna().sum()), 'кейсов': int(ratings.loc[started, 'case_id'].nunique()), 'номера кейсов': as_ranges(ratings.loc[started, 'case_id'].map(number_by_case).unique()), 'пустых ячеек в начатых кейсах': int(ratings.loc[started, name].isna().sum())}
pd.DataFrame.from_dict(coverage, orient='index')


Сетка: 3408 оценок, одинаковая у всех 6 судей


,оценок,кейсов,номера кейсов,пустых ячеек в начатых кейсах
expert_01,3406,60,1–60,2
expert_02,3408,60,1–60,0
expert_03,3407,60,1–60,1
expert_04,3408,60,1–60,0


In [6]:
rng = np.random.default_rng(SEED)

def kappa_from_tables(tables) -> np.ndarray:
    n11, n10, n01, n00 = np.moveaxis(np.asarray(tables, dtype=float), -1, 0)
    n = n11 + n10 + n01 + n00
    p_observed = (n11 + n00) / n
    p_a, p_b = ((n11 + n10) / n, (n11 + n01) / n)
    p_expected = p_a * p_b + (1 - p_a) * (1 - p_b)
    return (p_observed - p_expected) / (1 - p_expected)

def agreement(data: pd.DataFrame, a: str, b: str) -> dict:
    pair = data[['case_id', a, b]].dropna()
    x, y = (pair[a].astype(int), pair[b].astype(int))
    per_case = pd.DataFrame({'case_id': pair['case_id'], 'n11': x * y, 'n10': x * (1 - y), 'n01': (1 - x) * y, 'n00': (1 - x) * (1 - y)}).groupby('case_id').sum().to_numpy()
    kappa = float(kappa_from_tables(per_case.sum(axis=0)))
    assert np.isclose(kappa, cohen_kappa_score(x, y))
    n_cases = len(per_case)
    weights = rng.multinomial(n_cases, np.full(n_cases, 1 / n_cases), size=N_BOOT)
    low, high = np.nanpercentile(kappa_from_tables(weights @ per_case), [2.5, 97.5])
    return {'κ': kappa, '95% ДИ': f'{low:.3f}–{high:.3f}', 'согласие, %': 100 * float((x == y).mean()), 'оценок': len(pair), 'кейсов': n_cases}


In [7]:
has_expert = ratings[EXPERTS].notna().any(axis=1)
llm_vs_llm = pd.DataFrame.from_dict({'все 60 кейсов': agreement(ratings, *LLMS), 'кейсы с оценками экспертов': agreement(ratings[has_expert], *LLMS)}, orient='index')
llm_vs_llm.round(ROUND)


,κ,95% ДИ,"согласие, %",оценок,кейсов
все 60 кейсов,0.804,0.777–0.829,90.3,3408,60
кейсы с оценками экспертов,0.804,0.777–0.829,90.3,3408,60


In [8]:
llm_vs_expert = pd.DataFrame.from_dict({(judge, expert): agreement(ratings, judge, expert) for judge in LLMS for expert in EXPERTS}, orient='index')
llm_vs_expert.index.names = ['LLM', 'эксперт']
llm_vs_expert.round(ROUND)


κ       95% ДИ  согласие, %  оценок  кейсов
LLM         эксперт                                                   
gpt4.1      expert_01  0.653  0.615–0.691         82.8    3406      60
            expert_02  0.616  0.573–0.656         80.8    3408      60
            expert_03  0.648  0.616–0.678         83.1    3407      60
            expert_04  0.730  0.702–0.757         86.8    3408      60
gpt4.1-mini expert_01  0.615  0.575–0.653         80.9    3406      60
            expert_02  0.583  0.541–0.624         79.2    3408      60
            expert_03  0.585  0.543–0.621         79.8    3407      60
            expert_04  0.860  0.842–0.878         93.1    3408      60

In [9]:
n_experts = ratings[EXPERTS].notna().sum(axis=1)
yes_share = ratings[EXPERTS].mean(axis=1)
MAJORITY = 'большинство экспертов'
ratings[MAJORITY] = np.select([yes_share > 0.5, yes_share < 0.5], [1.0, 0.0], default=np.nan)
rated = n_experts > 0
print('Сколько экспертов за оценкой:', n_experts[rated].value_counts().sort_index().to_dict())
print(f'Ничьих исключено: {int((yes_share[rated] == 0.5).sum())} из {int(rated.sum())} оценок')
rows = {}
for judge in LLMS:
    rows[judge, 'большинство: все доступные эксперты'] = agreement(ratings, judge, MAJORITY)
    rows[judge, 'большинство: только оценки с ≥ 3 экспертами'] = agreement(ratings[n_experts >= 3], judge, MAJORITY)
    rows[judge, 'среднее попарных κ из п. 7'] = {'κ': llm_vs_expert.loc[judge, 'κ'].mean()}
llm_vs_majority = pd.DataFrame.from_dict(rows, orient='index')
llm_vs_majority.index.names = ['LLM', 'эксперты']
llm_vs_majority.astype({'оценок': 'Int64', 'кейсов': 'Int64'}).round(ROUND)


Сколько экспертов за оценкой: {3: 3, 4: 3405}
Ничьих исключено: 360 из 3408 оценок


κ       95% ДИ  \
LLM         эксперты                                                          
gpt4.1      большинство: все доступные эксперты          0.802  0.774–0.829   
            большинство: только оценки с ≥ 3 экспертами  0.802  0.774–0.829   
            среднее попарных κ из п. 7                   0.662          NaN   
gpt4.1-mini большинство: все доступные эксперты          0.788  0.753–0.822   
            большинство: только оценки с ≥ 3 экспертами  0.788  0.754–0.822   
            среднее попарных κ из п. 7                   0.661          NaN   

                                                         согласие, %  оценок  \
LLM         эксперты                                                           
gpt4.1      большинство: все доступные эксперты                 90.3    3048   
            большинство: только оценки с ≥ 3 экспертами         90.3    3048   
            среднее попарных κ из п. 7                           NaN    <NA>   
gpt4.1-mini большинство: все доступные эксперты                 89.5    3048   
            большинство: только оценки с ≥ 3 экспертами         89.5    3048   
            среднее попарных κ из п. 7                           NaN    <NA>   

                                                         кейсов  
LLM         эксперты                                             
gpt4.1      большинство: все доступные эксперты              60  
            большинство: только оценки с ≥ 3 экспертами      60  
            среднее попарных κ из п. 7                     <NA>  
gpt4.1-mini большинство: все доступные эксперты              60  
            большинство: только оценки с ≥ 3 экспертами      60  
            среднее попарных κ из п. 7                     <NA>